# TryOnGAN — Kaggle Training Notebook

**Paper**: _TryOnGAN: Body-Aware Try-On via StyleGAN_ (Lewis et al., 2021)

## Paper → Code Mapping
| Paper Section | File | Status |
|---|---|---|
| §3.1 StyleGAN2 Generator | `models/generator.py` | ✅ |
| §3.1 Pose/Cloth Encoding | `models/encoder.py` | ✅ |
| §3.2 Seg-Aware Discriminator | `models/discriminator.py` | ✅ |
| §3.3 Try-On Synthesis | `inference.py` | ✅ |
| §4 Adversarial + R1 + Perc | `losses.py` | ✅ |
| §5 Training Procedure | `train.py` | ✅ |

## Hardware Guide
| Setup | VRAM | RAM | Est. Time (100 epochs) |
|---|---|---|---|
| 1× T4 (batch=4) | ~10 GB | ~20 GB | 75–100 hrs |
| 2× T4 DDP (batch=4/gpu) | ~10 GB each | ~20 GB | 40–55 hrs |

In [ ]:
# ── 1. Install dependencies ─────────────────────────────────────────────
!pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q Pillow numpy

In [ ]:
# ── 2. GPU check ────────────────────────────────────────────────────────
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory/1e9:.1f} GB")

In [ ]:
# ── 3. Unzip VITON-HD dataset ────────────────────────────────────────────
# Attach VITON-HD from Kaggle datasets UI, then:
import os
DATA_ROOT = "/kaggle/input/viton-hd"  # adjust to your dataset name
print(f"Dataset exists: {os.path.exists(DATA_ROOT)}")
!ls {DATA_ROOT}/train/ 2>/dev/null || echo 'Check dataset path'

In [ ]:
# ── 4. Copy code to working dir ──────────────────────────────────────────
import shutil, os
SRC = "/kaggle/input/tryon-gan-code"   # your code dataset
DST = "/kaggle/working/tryon_gan"
if os.path.exists(SRC):
    shutil.copytree(SRC, DST, dirs_exist_ok=True)
    print("Code copied.")
else:
    print("Upload code as a Kaggle dataset or paste it directly.")
os.chdir(DST)

In [ ]:
# ── 5a. Single GPU training ──────────────────────────────────────────────
!python train.py \
    --data_root    /kaggle/input/viton-hd \
    --output_dir   /kaggle/working/tryon_out \
    --img_size     512 \
    --batch_size   4 \
    --grad_accum   4 \
    --epochs       100 \
    --use_compile \
    --log_every    100 \
    --save_every   5

In [ ]:
# ── 5b. 2× GPU Distributed Training ──────────────────────────────────────
# Make sure Kaggle accelerator is set to 2× T4
!torchrun \
    --nproc_per_node=2 \
    train.py \
    --data_root    /kaggle/input/viton-hd \
    --output_dir   /kaggle/working/tryon_out \
    --img_size     512 \
    --batch_size   4 \
    --grad_accum   4 \
    --epochs       100 \
    --use_compile \
    --log_every    50 \
    --save_every   5

In [ ]:
# ── 6. Monitor training samples ──────────────────────────────────────────
from IPython.display import display
from PIL import Image
from pathlib import Path

samples = sorted(Path("/kaggle/working/tryon_out").glob("samples_*.png"))
if samples:
    print(f"Latest: {samples[-1].name}")
    display(Image.open(samples[-1]))
else:
    print("No samples yet — training still starting up.")

In [ ]:
# ── 7. Inference: try on a new garment ───────────────────────────────────
CKPT      = "/kaggle/working/tryon_out/ckpt_epoch099.pt"
PERSON    = "/kaggle/input/viton-hd/test/image/some_person.jpg"
CLOTH     = "/kaggle/input/viton-hd/test/cloth/new_garment.jpg"
POSE_JSON = "/kaggle/input/viton-hd/test/openpose-json/some_person_keypoints.json"

!python inference.py \
    --ckpt     {CKPT} \
    --person   {PERSON} \
    --cloth    {CLOTH} \
    --pose     {POSE_JSON} \
    --out      /kaggle/working/tryon_result.jpg

display(Image.open("/kaggle/working/tryon_result.jpg"))